# SWITCH — Person 3: End-to-End ML Notebook
**Goal:** download and inspect the public CASAS smart-home sensor dataset, preprocess sensor-event windows, train an activity-context classifier, evaluate it on held-out homes, and save model + metrics.

**Important scope boundary:** CASAS contains activity annotations, not SWITCH-specific routine-conflict/user-correction/accepted-repair labels. This notebook trains an **activity-context classifier**, not a routine safety detector. Its predictions must never override deterministic regression assertions or user confirmation.

**How to use:** run the cells from top to bottom in Jupyter Notebook or VS Code. The first install cell may need internet access. Dataset download can take time depending on connection.

**Parser note:** If parsing fails, the updated parser prints sample lines from the downloaded archive so the format can be diagnosed instead of silently assuming a file layout.


## 1. Install dependencies
Run once in this notebook kernel. If installation fails due to low disk space, free space or use a drive with enough space before continuing.

In [ ]:
%pip install -q pandas numpy scikit-learn joblib matplotlib seaborn requests

## 2. Imports and reproducibility

In [ ]:
from pathlib import Path
from collections import Counter
import json, hashlib, zipfile, re, time, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

RANDOM_SEED = 42
WINDOW_EVENTS = 30
MIN_CLASS_COUNT = 10

ROOT = Path.cwd() / "switch_ml_work"
RAW_DIR = ROOT / "data" / "casas_raw"
EXTRACT_DIR = RAW_DIR / "extracted"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
for p in (RAW_DIR, EXTRACT_DIR, MODEL_DIR, REPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT.resolve())
print("Window events:", WINDOW_EVENTS)

## 3. Download CASAS dataset from Zenodo
This downloads the public archive and verifies the publisher-provided MD5 checksum when available. If you already have the archive, place `new_labeled_data.zip` in `switch_ml_work/data/casas_raw/` and skip the download cell.

In [ ]:
RECORD_ID = "17180309"
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"
ARCHIVE = RAW_DIR / "new_labeled_data.zip"

if not ARCHIVE.exists():
    response = requests.get(API_URL, timeout=60)
    response.raise_for_status()
    metadata = response.json()
    entry = next((f for f in metadata.get("files", [])
                  if f.get("key") == "new_labeled_data.zip"), None)
    if entry is None:
        raise RuntimeError(
            f"Archive `new_labeled_data.zip` not found in Zenodo record {RECORD_ID}. "
            "Inspect the record's file list and update the filename."
        )
    download_url = entry.get("links", {}).get("self")
    if not download_url:
        raise RuntimeError("Zenodo did not provide a direct download URL.")
    print("Downloading CASAS archive. This may take a few minutes...")
    h = hashlib.md5()
    with requests.get(download_url, stream=True, timeout=(30, 180)) as r:
        r.raise_for_status()
        with open(ARCHIVE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    h.update(chunk)
    expected = entry.get("checksum", "")
    if expected.startswith("md5:") and h.hexdigest().lower() != expected.split(":", 1)[1].lower():
        ARCHIVE.unlink(missing_ok=True)
        raise RuntimeError("Downloaded archive checksum mismatch; removed the incomplete archive.")
    print("Download complete. MD5:", h.hexdigest())
else:
    print("Using existing archive:", ARCHIVE)

with zipfile.ZipFile(ARCHIVE) as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f"Archive integrity check failed at: {bad_member}")
    print("Archive entries:", len(zf.namelist()))
    print("First entries:", zf.namelist()[:20])

## 4. Extract and inspect the actual data files
We inspect filenames and a few sample lines before parsing, because public datasets sometimes vary in layout.

In [ ]:
if not any(EXTRACT_DIR.iterdir()):
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(EXTRACT_DIR)

data_files = [
    p for p in EXTRACT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".txt", ".data", ".csv", ".ann"}
]
print("Candidate data files:", len(data_files))
for path in data_files[:8]:
    print("\\n---", path.relative_to(EXTRACT_DIR), "---")
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            for _ in range(4):
                line = f.readline()
                if not line:
                    break
                print(line.rstrip()[:220])
    except OSError as e:
        print("Could not read:", e)

## 5. Parse sensor events and activity labels
The parser supports common timestamped CASAS rows and embedded begin/end activity annotations. If this cell reports no rows or no labels, stop and inspect the sample lines above rather than inventing labels.

In [ ]:
# Robust CASAS parser: supports date + time columns, combined timestamps,
# comma/tab/space-delimited rows, and sensor files with optional headers.
from pathlib import Path
import pandas as pd
import numpy as np
import re

def _normalise_col(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

def parse_casas_file(path, home_id=None):
    path = Path(path)
    home_id = str(home_id or path.parent.name or path.stem)
    rows = []
    # Read with replacement for unusual encodings; do not assume a delimiter.
    try:
        raw_lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError:
        return pd.DataFrame()

    for line_no, raw in enumerate(raw_lines, start=1):
        line = raw.strip().lstrip("\ufeff")
        if not line or line.startswith(("#", "//", "%")):
            continue

        # First attempt: standard CASAS layout: date, time, sensor, message.
        # CASAS files commonly use whitespace, tabs, or commas.
        if "," in line:
            parts = [x.strip().strip('"').strip("'") for x in line.split(",")]
        elif "\t" in line:
            parts = [x.strip().strip('"').strip("'") for x in line.split("\t")]
        else:
            parts = re.split(r"\s+", line)

        parts = [x for x in parts if x != ""]
        if len(parts) < 3:
            continue

        # Skip headers.
        joined = " ".join(parts[:4]).lower()
        if ("sensor" in joined and ("date" in joined or "time" in joined)) or parts[0].lower() in {"date", "timestamp", "datetime"}:
            continue

        ts = pd.NaT
        sensor_idx = None
        message_idx = None

        # Layout A: separate date and time columns.
        if len(parts) >= 4:
            ts_candidate = pd.to_datetime(parts[0] + " " + parts[1], errors="coerce")
            if not pd.isna(ts_candidate):
                ts = ts_candidate
                sensor_idx, message_idx = 2, 3

        # Layout B: one timestamp column (ISO-8601 or quoted datetime).
        if pd.isna(ts):
            ts_candidate = pd.to_datetime(parts[0], errors="coerce")
            if not pd.isna(ts_candidate) and len(parts) >= 3:
                ts = ts_candidate
                sensor_idx, message_idx = 1, 2

        if pd.isna(ts) or sensor_idx is None or sensor_idx >= len(parts):
            continue

        sensor = str(parts[sensor_idx]).strip()
        message = str(parts[message_idx]).strip() if message_idx is not None and message_idx < len(parts) else ""
        if not sensor or sensor.lower() in {"sensor", "sensorid", "sensor_id"}:
            continue

        # Optional activity/task label fields, when present in this dataset format.
        activity = ""
        extra = parts[message_idx + 1:] if message_idx is not None else []
        if extra:
            candidate = " ".join(str(x).strip() for x in extra).strip()
            if candidate.lower() not in {"", "-", "none", "nan", "null", "unknown"}:
                activity = candidate

        rows.append({
            "timestamp": ts, "home_id": home_id, "sensor": sensor,
            "message": message.upper(), "sensor_type": "unknown",
            "activity": activity, "source_file": path.name, "source_line": line_no
        })

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    # Standardize obvious activity annotations when present as part of a row.
    frame["activity"] = frame["activity"].fillna("").astype(str).str.strip()
    return frame

# Include common plain-text dataset extensions and exclude obvious non-data files.
data_files = [
    p for p in EXTRACT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {
        ".txt", ".data", ".csv", ".ann", ".log", ".dat", ".tsv"
    }
]
print("Candidate data files:", len(data_files))
if not data_files:
    print("Archive files (first 40):")
    with zipfile.ZipFile(ARCHIVE) as zf:
        print("\n".join(zf.namelist()[:40]))
    raise RuntimeError("No supported text data files found in the archive. Inspect the archive listing above.")

# Infer household IDs correctly.
# BUG FIX: path.parent.name gives every flat archive file the SAME home ID,
# so GroupShuffleSplit sees only one group and crashes. If files live in
# multiple household folders, use the relative parent folder; if all files
# are in one folder, use each data file's stem as its household/file group.
relative_parents = {str(p.relative_to(EXTRACT_DIR).parent) for p in data_files}
use_parent_as_home = len(relative_parents) > 1 and any(
    str(p.relative_to(EXTRACT_DIR).parent) not in {".", ""} for p in data_files
)

def infer_home_id(path):
    rel = path.relative_to(EXTRACT_DIR)
    parent = str(rel.parent)
    if use_parent_as_home and parent not in {".", ""}:
        return parent.replace("\\", "/")
    return path.stem

frames, parse_errors = [], []
for path in data_files:
    try:
        inferred_home = infer_home_id(path)
        frame = parse_casas_file(path, home_id=inferred_home)
        if not frame.empty:
            frames.append(frame)
    except (OSError, UnicodeError, ValueError, TypeError) as e:
        parse_errors.append((str(path), str(e)))

if not frames:
    print("\nParser could not read rows. Example lines from the first candidate file:")
    for path in data_files[:2]:
        print(f"\n--- {path.relative_to(EXTRACT_DIR)} ---")
        try:
            for line in path.read_text(encoding="utf-8", errors="replace").splitlines()[:12]:
                print(repr(line[:240]))
        except OSError as e:
            print("Could not read:", e)
    raise RuntimeError(
        "No sensor rows were parsed. The diagnostic examples above show the actual file format; "
        "do not proceed to training until the parser matches it."
    )

events = pd.concat(frames, ignore_index=True)
events = events.sort_values(["home_id", "timestamp", "source_line"]).reset_index(drop=True)
labelled_rows = int(events["activity"].astype(str).str.len().gt(0).sum())

print("Parsed event rows:", len(events))
print("Homes/files grouped:", events["home_id"].nunique())
print("Rows with activity labels in event files:", labelled_rows)
print("Distinct activity labels:", events.loc[events.activity.astype(str).str.len() > 0, "activity"].nunique())
print("Parser errors skipped:", len(parse_errors))
display(events.head(10))

if labelled_rows == 0:
    raise RuntimeError(
        "Sensor events parsed, but no activity labels were found in these event files. "
        "This dataset may store annotations in separate files. Inspect the candidate-file previews "
        "from the earlier cell and align labels explicitly; do not fabricate labels."
    )


## 6. Explore label coverage

In [ ]:
label_counts = events.loc[events["activity"].astype(str).str.len() > 0, "activity"].value_counts()
display(label_counts.head(25).rename_axis("activity").to_frame("event_rows"))
ax = label_counts.head(20).sort_values().plot(kind="barh", figsize=(9, 6), title="Most frequent annotated activities")
ax.set_xlabel("Annotated sensor events")
plt.tight_layout()
plt.show()

## 7. Build non-overlapping event-window features
Each row represents a fixed-size window of sensor events. Features include sensor/message counts, time-of-day, weekday, mean inter-event gap and sensor diversity. A window receives the most common activity label observed inside it.

In [ ]:
def build_event_windows(data, window_events=30):
    feature_rows, labels, groups = [], [], []
    for home, home_data in data.groupby("home_id", sort=True):
        home_data = home_data.sort_values("timestamp").reset_index(drop=True)
        for end in range(window_events, len(home_data) + 1, window_events):
            win = home_data.iloc[end-window_events:end]
            counts = Counter((win["sensor"].astype(str) + "=" + win["message"].astype(str)).tolist())
            row = {f"event::{k}": int(v) for k, v in counts.items()}
            last_ts = win.iloc[-1]["timestamp"]
            row["hour"] = int(last_ts.hour)
            row["weekday"] = int(last_ts.dayofweek)
            gaps = win["timestamp"].diff().dt.total_seconds().to_numpy()[1:]
            row["mean_gap_seconds"] = float(np.clip(gaps, 0, None).mean()) if len(gaps) else 0.0
            row["unique_sensors"] = int(win["sensor"].nunique())

            nonempty = win.loc[win["activity"].astype(str).str.len() > 0, "activity"].astype(str)
            label = nonempty.mode().iloc[0] if not nonempty.empty else ""
            feature_rows.append(row)
            labels.append(label)
            groups.append(home)

    X = pd.DataFrame(feature_rows).fillna(0)
    y = pd.Series(labels, name="activity")
    groups = pd.Series(groups, name="home_id")
    return X, y, groups

X_all, y_all, groups_all = build_event_windows(events, WINDOW_EVENTS)
label_mask = y_all.astype(str).str.len() > 0
X, y, groups = X_all.loc[label_mask].reset_index(drop=True), y_all.loc[label_mask].reset_index(drop=True), groups_all.loc[label_mask].reset_index(drop=True)

class_counts = y.value_counts()
eligible_classes = class_counts[class_counts >= MIN_CLASS_COUNT].index
keep = y.isin(eligible_classes)
X, y, groups = X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True), groups.loc[keep].reset_index(drop=True)

print("Total windows:", len(X_all))
print("Labeled windows before class filtering:", int(label_mask.sum()))
print("Windows after minimum class count filter:", len(X))
print("Classes retained:", y.nunique())
print("Homes represented:", groups.nunique())
print("Home window counts:")
print(groups.value_counts().to_string())
display(y.value_counts().rename_axis("activity").to_frame("windows"))
if groups.nunique() < 2:
    raise RuntimeError(
        "Only one home/file group remains after filtering. The home-held-out evaluation "
        "requires at least two independently grouped homes. Check the inferred home IDs and "
        "whether the archive contains multiple home-specific files; do not report test metrics."
    )
if y.nunique() < 2:
    raise RuntimeError(
        "Fewer than two activities have enough labelled windows. Try MIN_CLASS_COUNT = 3, "
        "or inspect annotation coverage. Do not proceed with a meaningless one-class model."
    )

## 8. Train/test split by home, then train Random Forest
A home-held-out split reduces leakage from the same household appearing in both train and test. With only a few homes, results are preliminary and can vary with the split.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
train_homes = sorted(groups.iloc[train_idx].unique().tolist())
test_homes = sorted(groups.iloc[test_idx].unique().tolist())

if not train_homes or not test_homes:
    raise RuntimeError("Could not form a home-held-out split. Need data from multiple homes.")

vectorizer = DictVectorizer(sparse=True)
X_records = X.to_dict(orient="records")
X_train = vectorizer.fit_transform([X_records[i] for i in train_idx])
X_test = vectorizer.transform([X_records[i] for i in test_idx])

model = RandomForestClassifier(
    n_estimators=180,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model.fit(X_train, y.iloc[train_idx])
pred = model.predict(X_test)

accuracy = accuracy_score(y.iloc[test_idx], pred)
macro_f1 = f1_score(y.iloc[test_idx], pred, average="macro", zero_division=0)
print("Train homes:", train_homes)
print("Test homes:", test_homes)
print(f"Test accuracy: {accuracy:.4f}")
print(f"Test macro-F1: {macro_f1:.4f}")
print("\\nClassification report:")
print(classification_report(y.iloc[test_idx], pred, zero_division=0))

## 9. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y.iloc[test_idx], pred, xticks_rotation=90, normalize=None,
    cmap=None, ax=ax, colorbar=False
)
ax.set_title("CASAS activity-context classification — held-out homes")
plt.tight_layout()
plt.show()

## 10. Save model, vectorizer, and evaluation report

In [ ]:
model_path = MODEL_DIR / "casas_activity_context_random_forest.joblib"
vectorizer_path = MODEL_DIR / "casas_feature_vectorizer.joblib"
metrics_path = REPORT_DIR / "casas_activity_context_metrics.json"

joblib.dump(model, model_path)
joblib.dump(vectorizer, vectorizer_path)

report = classification_report(
    y.iloc[test_idx], pred, output_dict=True, zero_division=0
)
metrics = {
    "dataset": "CASAS Smart Home public dataset, Zenodo record 17180309",
    "task": "activity-context classification",
    "source_archive": ARCHIVE.name,
    "source_archive_md5": hashlib.md5(ARCHIVE.read_bytes()).hexdigest(),
    "parsed_sensor_events": int(len(events)),
    "labelled_source_rows": int(labelled_rows),
    "window_events": int(WINDOW_EVENTS),
    "minimum_class_count": int(MIN_CLASS_COUNT),
    "n_windows_after_filter": int(len(X)),
    "classes": sorted(y.unique().tolist()),
    "train_homes": train_homes,
    "test_homes": test_homes,
    "test_accuracy": float(accuracy),
    "test_macro_f1": float(macro_f1),
    "classification_report": report,
    "limitations": [
        "Predicts source activity annotations, not user corrections or routine conflicts.",
        "A single held-out-home split over a small number of homes is preliminary.",
        "This is not a safety model and must not override deterministic SWITCH requirements or user confirmation."
    ]
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved model:", model_path.resolve())
print("Saved vectorizer:", vectorizer_path.resolve())
print("Saved metrics:", metrics_path.resolve())
print("Archive MD5:", metrics["source_archive_md5"])

## 11. Inference example on a held-out window

In [ ]:
sample_index = int(test_idx[0])
sample_record = X_records[sample_index]
sample_vector = vectorizer.transform([sample_record])
predicted_activity = model.predict(sample_vector)[0]
actual_activity = y.iloc[sample_index]
print("Predicted activity:", predicted_activity)
print("Actual activity:", actual_activity)
print("This prediction is activity context only; it does not determine PASS/FAIL for a routine.")

## 12. What this achieves — and what remains for SWITCH

**Completed by this notebook when all cells run successfully**
- Downloads and validates the real public CASAS archive.
- Parses timestamped sensor events and available activity annotations.
- Builds event-window features.
- Trains a Random Forest activity-context classifier.
- Evaluates on held-out home(s), reports accuracy, macro-F1, per-class metrics and confusion matrix.
- Saves model/vectorizer/metrics artifacts.

**Not solved by CASAS**
CASAS does not provide labels for a user-confirmed household requirement, a routine version that violates it, or whether a proposed repair is valid. Therefore it cannot directly train a supervised SWITCH regression/conflict detector. Keep the deterministic regression engine as the authority for PASS/FAIL. To train a SWITCH-specific model, build a separately reviewed, versioned benchmark of routine scenarios and expected outcomes; do not label ordinary sensor events as safety failures.


## Package-based inference and service readiness (Person 3C completion)
These cells call the package implementation and existing artifacts. They do not retrain by default. Synthetic artifacts are not evidence of real CASAS accuracy.

In [ ]:
from pathlib import Path
import json, os, sys
PROJECT = Path.cwd()
if not (PROJECT / 'switch_p3c').exists():
    PROJECT = PROJECT.parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from switch_p3c.casas.inference import ActivityModel
model = ActivityModel.load(PROJECT / 'switch_p3c' / 'artifacts')
print({'ready': model.ready, 'error': model.error or None, 'window_events': model.window_events})

In [ ]:
metrics_path = PROJECT / 'switch_p3c' / 'artifacts' / 'casas_activity_context_metrics.json'
if metrics_path.exists():
    saved_metrics = json.loads(metrics_path.read_text())
    print(json.dumps({'data_source': saved_metrics.get('data_source'), 'held_out_test': saved_metrics.get('held_out_test'), 'data_source_warning': saved_metrics.get('data_source_warning')}, indent=2))
else:
    print('Metrics file missing; no accuracy claim can be made.')

### Optional training (explicit opt-in only)
To train, first prepare a real labelled CASAS dataset in the documented format and set `SWITCH_CASAS_DATA_DIR`. Do not use synthetic data for real-world accuracy claims. The trainer enforces home-held-out splits and writes metrics/artifacts.

In [ ]:
# This cell does not train unless the caller explicitly sets SWITCH_RUN_TRAINING=1.
if os.getenv('SWITCH_RUN_TRAINING') == '1':
    data_dir = os.getenv('SWITCH_CASAS_DATA_DIR')
    data_source = os.getenv('SWITCH_CASAS_DATA_SOURCE', 'casas_real')
    if not data_dir: raise ValueError('Set SWITCH_CASAS_DATA_DIR to the prepared dataset directory')
    from switch_p3c.casas.train import run
    trained = run(data_dir, str(PROJECT / 'switch_p3c' / 'artifacts'), data_source=data_source)
    print(json.dumps({'data_source': trained['data_source'], 'held_out_test': trained['held_out_test']}, indent=2))
else:
    print('Training skipped. Set SWITCH_RUN_TRAINING=1 and SWITCH_CASAS_DATA_DIR to opt in.')